<a href="https://colab.research.google.com/github/Pujitha-pitta/Flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



**Lane**

My lane is **content-refresh prioritisation**.

**Baseline rule**

I prioritise content pages for review when they:

1. have not been updated recently,
2. have CTR below the typical CTR of pages in a similar search-position bucket, and
3. receive enough historical impressions for an improvement to have practical value.

The score is a decision-support baseline, not proof that a page definitely needs refreshing. A human should review the page, search intent, seasonality, and current content before taking action.

**Allowed inputs**

- Days since the last update
- Historical CTR
- Historical average position
- Historical impressions

**Inputs intentionally excluded**

- Product-generated refresh flags
- Trend labels used as targets
- Future performance windows
- Client names
- URLs
- Private search queries

In [3]:
from pathlib import Path

csv_files = list(Path(".").rglob("*.csv"))

print(f"Found {len(csv_files)} CSV file(s):\n")

for file in csv_files:
    print(file)

Found 4 CSV file(s):

sample_data/mnist_test.csv
sample_data/california_housing_train.csv
sample_data/mnist_train_small.csv
sample_data/california_housing_test.csv


In [4]:
!pwd
!ls

/content
sample_data


In [5]:
!git clone https://github.com/Pujitha-pitta/Flyrank-ML-internship.git

Cloning into 'Flyrank-ML-internship'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 143 (delta 50), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.87 MiB | 9.99 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [6]:
%cd /content/Flyrank-ML-internship

/content/Flyrank-ML-internship


In [7]:
!ls
!find . -name "*.csv"

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission
./data/raw/content_refresh_anonymized.csv
./outputs/refresh_queue_sample.csv


In [8]:
import pandas as pd
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

print("File exists:", data_path.exists())

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

File exists: True
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [9]:
print("Rows:", len(df))
print(df.columns.tolist())

Rows: 30000
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

CONTENT_ID_COL = "content_id"
STALENESS_COL = "days_since_last_update"
CTR_COL = "ctr"
POSITION_COL = "avg_position"
IMPRESSION_COL = "impressions_90d"

required_columns = [
    CONTENT_ID_COL,
    STALENESS_COL,
    CTR_COL,
    POSITION_COL,
    IMPRESSION_COL
]

baseline_df = df[required_columns].copy()

for col in [
    STALENESS_COL,
    CTR_COL,
    POSITION_COL,
    IMPRESSION_COL
]:
    baseline_df[col] = pd.to_numeric(
        baseline_df[col],
        errors="coerce"
    )

baseline_df = baseline_df.dropna(subset=required_columns).copy()

baseline_df = baseline_df[
    (baseline_df[STALENESS_COL] >= 0)
    & (baseline_df[CTR_COL] >= 0)
    & (baseline_df[POSITION_COL] > 0)
    & (baseline_df[IMPRESSION_COL] >= 0)
].copy()

print("Usable rows:", len(baseline_df))
baseline_df.head()

Usable rows: 28795


,content_id,days_since_last_update,ctr,avg_position,impressions_90d
0,content_304f48230142,20,0.76,10.6,3803
1,content_a1fb4e703a9e,25,0.05,20.3,15320
2,content_9aa793d4d895,20,0.09,36.5,12581
3,content_331d6c4de07b,22,0.49,6.2,11751
4,content_d99b7a2d90ca,14,0.13,44.0,19140


In [11]:
baseline_df["staleness_bucket"] = pd.cut(
    baseline_df[STALENESS_COL],
    bins=[-1, 90, 180, 365, np.inf],
    labels=[
        "0–90 days",
        "91–180 days",
        "181–365 days",
        "365+ days"
    ]
)

staleness_table = (
    baseline_df
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=(CONTENT_ID_COL, "size"),
        mean_ctr=(CTR_COL, "mean"),
        median_ctr=(CTR_COL, "median"),
        median_impressions=(IMPRESSION_COL, "median")
    )
    .reset_index()
)

staleness_table

,staleness_bucket,n,mean_ctr,median_ctr,median_impressions
0,0–90 days,19475,0.624817,0.07,577.0
1,91–180 days,9162,0.238601,0.10,1700.0
2,181–365 days,153,3.328758,0.00,21.0
3,365+ days,5,20.000000,0.00,2.0


In [12]:
baseline_df["position_bucket"] = pd.cut(
    baseline_df[POSITION_COL],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=[
        "Top 3",
        "4–10",
        "11–20",
        "21–50",
        "50+"
    ],
    include_lowest=True
)

position_table = (
    baseline_df
    .groupby("position_bucket", observed=True)
    .agg(
        n=(CONTENT_ID_COL, "size"),
        mean_ctr=(CTR_COL, "mean"),
        median_ctr=(CTR_COL, "median"),
        median_impressions=(IMPRESSION_COL, "median")
    )
    .reset_index()
)

position_table

,position_bucket,n,mean_ctr,median_ctr,median_impressions
0,Top 3,1141,2.714303,0.00,74.0
1,4–10,11842,0.651045,0.16,1184.0
2,11–20,7273,0.323443,0.10,870.0
3,21–50,7225,0.222345,0.03,807.0
4,50+,1314,0.150784,0.00,219.5


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
expected_ctr = (
    baseline_df
    .groupby("position_bucket", observed=True)[CTR_COL]
    .median()
    .rename("expected_ctr")
)

baseline_df = baseline_df.join(
    expected_ctr,
    on="position_bucket"
)

baseline_df["ctr_gap"] = (
    baseline_df["expected_ctr"] - baseline_df[CTR_COL]
)

baseline_df[
    [
        CONTENT_ID_COL,
        "position_bucket",
        CTR_COL,
        "expected_ctr",
        "ctr_gap"
    ]
].head()

,content_id,position_bucket,ctr,expected_ctr,ctr_gap
0,content_304f48230142,11–20,0.76,0.10,-0.66
1,content_a1fb4e703a9e,21–50,0.05,0.03,-0.02
2,content_9aa793d4d895,21–50,0.09,0.03,-0.06
3,content_331d6c4de07b,4–10,0.49,0.16,-0.33
4,content_d99b7a2d90ca,21–50,0.13,0.03,-0.10


In [14]:
baseline_df["staleness_points"] = np.select(
    [
        baseline_df[STALENESS_COL] >= 365,
        baseline_df[STALENESS_COL] >= 180,
        baseline_df[STALENESS_COL] >= 90,
    ],
    [40, 30, 15],
    default=0,
)

baseline_df["ctr_gap_points"] = np.select(
    [
        baseline_df["ctr_gap"] >= 0.05,
        baseline_df["ctr_gap"] >= 0.02,
        baseline_df["ctr_gap"] > 0,
    ],
    [40, 25, 10],
    default=0,
)

impression_median = baseline_df[IMPRESSION_COL].median()
impression_q75 = baseline_df[IMPRESSION_COL].quantile(0.75)

baseline_df["volume_points"] = np.select(
    [
        baseline_df[IMPRESSION_COL] >= impression_q75,
        baseline_df[IMPRESSION_COL] >= impression_median,
    ],
    [20, 10],
    default=0,
)

baseline_df["baseline_action_score"] = (
    baseline_df["staleness_points"]
    + baseline_df["ctr_gap_points"]
    + baseline_df["volume_points"]
)

baseline_df[
    [
        "staleness_points",
        "ctr_gap_points",
        "volume_points",
        "baseline_action_score"
    ]
].head()

,staleness_points,ctr_gap_points,volume_points,baseline_action_score
0,0,0,10,10
1,0,0,20,20
2,0,0,20,20
3,0,0,20,20
4,0,0,20,20


In [15]:
is_stale = baseline_df[STALENESS_COL] >= 180
is_low_ctr = baseline_df["ctr_gap"] > 0
is_high_volume = baseline_df[IMPRESSION_COL] >= impression_median

baseline_df["reason_code"] = np.select(
    [
        is_stale & is_low_ctr & is_high_volume,
        is_stale & is_low_ctr,
        is_low_ctr & is_high_volume,
        is_stale,
    ],
    [
        "STALE_LOW_CTR_HIGH_VOLUME",
        "STALE_LOW_CTR",
        "LOW_CTR_HIGH_VOLUME",
        "STALE_ONLY",
    ],
    default="NO_BASELINE_TRIGGER",
)

baseline_df["reason_code"].value_counts()

,count
reason_code,
NO_BASELINE_TRIGGER,24777
LOW_CTR_HIGH_VOLUME,3860
STALE_LOW_CTR,92
STALE_ONLY,64
STALE_LOW_CTR_HIGH_VOLUME,2


In [16]:
baseline_df["action_label"] = np.select(
    [
        baseline_df["baseline_action_score"] >= 70,
        baseline_df["baseline_action_score"] >= 45,
    ],
    [
        "REFRESH_NOW",
        "REVIEW",
    ],
    default="MONITOR",
)

baseline_df["action_label"].value_counts()

,count
action_label,
MONITOR,24317
REVIEW,4050
REFRESH_NOW,428


In [17]:
baseline_df["confidence_note"] = np.select(
    [
        is_stale & is_low_ctr & is_high_volume & (baseline_df["ctr_gap"] >= 0.02),
        is_stale & is_low_ctr,
        is_low_ctr | is_stale,
    ],
    [
        "High confidence: multiple signals agree",
        "Medium confidence: staleness and CTR agree",
        "Low confidence: recommendation depends mainly on one signal",
    ],
    default="Low confidence: no strong trigger",
)

baseline_df["confidence_note"].head()

,confidence_note
0,Low confidence: no strong trigger
1,Low confidence: no strong trigger
2,Low confidence: no strong trigger
3,Low confidence: no strong trigger
4,Low confidence: no strong trigger


In [18]:
ranked_queue = (
    baseline_df
    .sort_values(
        by=[
            "baseline_action_score",
            IMPRESSION_COL,
            "ctr_gap"
        ],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0,
    "rank",
    np.arange(1, len(ranked_queue) + 1)
)

ranked_queue.head()

,rank,content_id,days_since_last_update,ctr,avg_position,impressions_90d,staleness_bucket,position_bucket,expected_ctr,ctr_gap,staleness_points,ctr_gap_points,volume_points,baseline_action_score,reason_code,action_label,confidence_note
0,1,content_55a5b1c46474,373,0.00,7.5,35,365+ days,4–10,0.16,0.16,40,40,0,80,STALE_LOW_CTR,REFRESH_NOW,Medium confidence: staleness and CTR agree
1,2,content_1b4ec72dafd4,372,0.00,7.0,2,365+ days,4–10,0.16,0.16,40,40,0,80,STALE_LOW_CTR,REFRESH_NOW,Medium confidence: staleness and CTR agree
2,3,content_36ff89c8214e,104,0.05,7.3,295097,91–180 days,4–10,0.16,0.11,15,40,20,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...
3,4,content_c8e9d6ab9013,104,0.00,9.7,208678,91–180 days,4–10,0.16,0.16,15,40,20,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...
4,5,content_a7427266c305,104,0.11,5.7,201111,91–180 days,4–10,0.16,0.05,15,40,20,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...


In [19]:
baseline_output = ranked_queue[
    [
        "rank",
        CONTENT_ID_COL,
        "baseline_action_score",
        "reason_code",
        "action_label",
        "confidence_note",
        STALENESS_COL,
        POSITION_COL,
        CTR_COL,
        "expected_ctr",
        "ctr_gap",
        IMPRESSION_COL
    ]
]

baseline_output.head(20)

,rank,content_id,baseline_action_score,reason_code,action_label,confidence_note,days_since_last_update,avg_position,ctr,expected_ctr,ctr_gap,impressions_90d
0,1,content_55a5b1c46474,80,STALE_LOW_CTR,REFRESH_NOW,Medium confidence: staleness and CTR agree,373,7.5,0.00,0.16,0.16,35
1,2,content_1b4ec72dafd4,80,STALE_LOW_CTR,REFRESH_NOW,Medium confidence: staleness and CTR agree,372,7.0,0.00,0.16,0.16,2
2,3,content_36ff89c8214e,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,7.3,0.05,0.16,0.11,295097
3,4,content_c8e9d6ab9013,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,9.7,0.00,0.16,0.16,208678
4,5,content_a7427266c305,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,5.7,0.11,0.16,0.05,201111
5,6,content_91652435f57a,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,7.8,0.06,0.16,0.10,159590
6,7,content_97a86caf3a3d,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,6.4,0.07,0.16,0.09,147670
7,8,content_c1fe78bc4e37,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,7.5,0.03,0.16,0.13,134055
8,9,content_4c76e9b13aea,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,7.4,0.07,0.16,0.09,127952
9,10,content_b115f7c74779,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,8.0,0.03,0.16,0.13,123469


In [20]:
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

baseline_output.to_csv(output_path, index=False)

print("Saved successfully!")
print(output_path)

Saved successfully!
work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
top20 = baseline_output.head(20)
top20

,rank,content_id,baseline_action_score,reason_code,action_label,confidence_note,days_since_last_update,avg_position,ctr,expected_ctr,ctr_gap,impressions_90d
0,1,content_55a5b1c46474,80,STALE_LOW_CTR,REFRESH_NOW,Medium confidence: staleness and CTR agree,373,7.5,0.00,0.16,0.16,35
1,2,content_1b4ec72dafd4,80,STALE_LOW_CTR,REFRESH_NOW,Medium confidence: staleness and CTR agree,372,7.0,0.00,0.16,0.16,2
2,3,content_36ff89c8214e,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,7.3,0.05,0.16,0.11,295097
3,4,content_c8e9d6ab9013,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,9.7,0.00,0.16,0.16,208678
4,5,content_a7427266c305,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,5.7,0.11,0.16,0.05,201111
5,6,content_91652435f57a,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,7.8,0.06,0.16,0.10,159590
6,7,content_97a86caf3a3d,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,6.4,0.07,0.16,0.09,147670
7,8,content_c1fe78bc4e37,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,7.5,0.03,0.16,0.13,134055
8,9,content_4c76e9b13aea,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,7.4,0.07,0.16,0.09,127952
9,10,content_b115f7c74779,75,LOW_CTR_HIGH_VOLUME,REFRESH_NOW,Low confidence: recommendation depends mainly ...,104,8.0,0.03,0.16,0.13,123469


Top-20 Review

The table above shows the 20 highest-ranked pages based on the baseline action score.

Each recommendation is based on historical staleness, CTR performance relative to similar search positions, and historical impressions.

These recommendations are intended as decision support rather than final decisions. Every page should be manually reviewed before taking action because factors such as seasonality, search intent, or recent content updates may explain the observed signals.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks**

Some recommendations may be weaker because they are driven mainly by one signal rather than multiple signals.

Examples include:

- A page that is old but still performs well.
- A page with a small CTR gap that may not justify a refresh.
- A page affected by seasonal search demand rather than content quality.

These pages should be manually reviewed before any action is taken.

**Leakage Check**

This baseline rule uses only historical information:

- Days since last update
- Historical CTR
- Historical average position
- Historical impressions

The rule does not use:

- trend_direction
- trend_pct
- Product-generated flags
- Future information
- Target labels

Therefore, no obvious leakage is present in the baseline score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.